In [2]:
# ==========================================================
# BLOQUE 1. Configuración
# Admisión Doctorado
# ==========================================================

import json
import requests

from bs4 import BeautifulSoup
from urllib.parse import urljoin


# ----------------------------------------------------------
# Configuración
# ----------------------------------------------------------

URL = (
    "https://www.upv.es/admision/admision-doctorado/index-es.html"
)

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0"
    )
}

from google.colab import drive

drive.mount("/content/drive")

# ----------------------------------------------------------
# Funciones auxiliares
# ----------------------------------------------------------

def texto_limpio(elemento):

    if elemento is None:
        return ""

    return " ".join(
        elemento.stripped_strings
    )


def url_absoluta(url):

    if not url:
        return ""

    return urljoin(
        URL,
        url
    )


def primer_enlace(elemento):

    if elemento is None:
        return ""

    enlace = elemento.find(
        "a",
        href=True
    )

    if enlace:
        return url_absoluta(
            enlace["href"]
        )

    return ""


def obtener_descripcion(contenedor):

    if contenedor is None:
        return ""

    parrafos = contenedor.find_all(
        "p",
        recursive=False
    )

    texto = " ".join(
        texto_limpio(p)
        for p in parrafos
    )

    return texto.strip()

Mounted at /content/drive


In [3]:
# ==========================================================
# BLOQUE 2. Ruta del JSON
# ==========================================================

import os


NOMBRE_PROGRAMA = "Extrae_Admision_Doctorado.ipynb"


ruta_programa = None

for root, dirs, files in os.walk("/content/drive/MyDrive"):

    if NOMBRE_PROGRAMA in files:

        ruta_programa = root
        break


if ruta_programa is None:

    raise Exception(
        "No se ha encontrado el notebook."
    )


CARPETA_JSON = os.path.join(
    ruta_programa,
    "JSONs"
)


os.makedirs(
    CARPETA_JSON,
    exist_ok=True
)


RUTA_JSON = os.path.join(
    CARPETA_JSON,
    "admision_doctorado.json"
)


print("Directorio del proyecto:")
print(ruta_programa)

print()

print("JSON:")
print(RUTA_JSON)

Directorio del proyecto:
/content/drive/MyDrive/TFG Teleco

JSON:
/content/drive/MyDrive/TFG Teleco/JSONs/admision_doctorado.json


In [4]:
# ==========================================================
# BLOQUE 4. Descarga y localización de secciones
# ==========================================================

respuesta = requests.get(
    URL,
    headers=HEADERS
)

respuesta.raise_for_status()

soup = BeautifulSoup(
    respuesta.text,
    "html.parser"
)


# ----------------------------------------------------------
# Padre
# ----------------------------------------------------------

titulo_padre = texto_limpio(
    soup.find("h1")
)

descripcion_padre = texto_limpio(
    soup.find("h2")
)


# ----------------------------------------------------------
# Secciones
# ----------------------------------------------------------

secciones = []

for i in range(1, 7):

    identificador = f"section-{i:02d}"

    seccion = soup.find(
        "section",
        id=identificador
    )

    if seccion is None:

        print(
            f"No encontrada {identificador}"
        )

        continue

    secciones.append(
        seccion
    )


# ----------------------------------------------------------
# Validación
# ----------------------------------------------------------

print("Título:", titulo_padre)
print("Descripción:", descripcion_padre)
print()

print(
    "Secciones encontradas:",
    len(secciones)
)

for s in secciones:

    titulo = s.find(
        ["h2", "h3"]
    )

    print(
        "-",
        s["id"],
        "|",
        texto_limpio(titulo)
    )

Título: Admisión a doctorado
Descripción: Explora lo desconocido, enfrenta los retos del mañana

Secciones encontradas: 6
- section-01 | Conoce la Escuela de Doctorado
- section-02 | Elige Programa de Doctorado y confirma tutor
- section-03 | Consulta las ayudas predoctorales
- section-04 | Presenta tu solicitud
- section-05 | Realiza la matrícula
- section-06 | Comienza tu aventura


In [5]:
# ==========================================================
# BLOQUE 5. Funciones de extracción
# ==========================================================

def extraer_tarjetas(seccion):

    tarjetas = []

    for caja in seccion.select(".box-number-box"):

        titulo = texto_limpio(
            caja.find("h3")
        )

        if not titulo:
            continue

        descripcion = obtener_descripcion(
            caja
        )

        url = primer_enlace(
            caja
        )

        tarjetas.append({
            "titulo": titulo,
            "descripcion": descripcion,
            "url": url
        })

    return tarjetas


def extraer_banners(seccion):

    banners = []

    for banner in seccion.select(".banner--content"):

        titulo = texto_limpio(
            banner.find("h3")
        )

        descripcion = obtener_descripcion(
            banner
        )

        url = primer_enlace(
            banner
        )

        banners.append({
            "titulo": titulo,
            "descripcion": descripcion,
            "url": url
        })

    return banners


def extraer_acordeones(seccion):

    acordeones = []

    for bloque in seccion.select(".accordion-element-content"):

        titulo = texto_limpio(
            bloque.find("h3")
        )

        texto = texto_limpio(
            bloque
        )

        enlaces = []

        for a in bloque.find_all(
            "a",
            href=True
        ):

            enlaces.append({

                "texto": texto_limpio(a),

                "url": url_absoluta(
                    a["href"]
                )

            })

        acordeones.append({

            "titulo": titulo,

            "texto": texto,

            "enlaces": enlaces

        })

    return acordeones


def extraer_enlaces(seccion):

    enlaces = []

    vistos = set()

    for a in seccion.find_all(
        "a",
        href=True
    ):

        texto = texto_limpio(a)

        url = url_absoluta(
            a["href"]
        )

        if not texto or url in vistos:

            continue

        vistos.add(url)

        enlaces.append({

            "texto": texto,

            "url": url

        })

    return enlaces

In [6]:
# ==========================================================
# BLOQUE 6. Construcción del JSON
# ==========================================================

padre = {

    "titulo": titulo_padre,

    "url": URL,

    "descripcion": descripcion_padre,

    "secciones": []

}


for seccion in secciones:


    titulo = texto_limpio(
        seccion.find(
            ["h2", "h3"]
        )
    )

    if not titulo:

        continue


    descripcion = obtener_descripcion(
        seccion
    )


    tarjetas = extraer_tarjetas(
        seccion
    )

    acordeones = extraer_acordeones(
        seccion
    )

    banners = extraer_banners(
        seccion
    )

    enlaces = extraer_enlaces(
        seccion
    )


    datos_seccion = {

        "id": seccion.get("id"),

        "titulo": titulo,

        "descripcion": descripcion,

        "tarjetas": tarjetas,

        "acordeones": acordeones,

        "banners": banners,

        "enlaces": enlaces

    }


    padre["secciones"].append(
        datos_seccion
    )


datos = {

    "padres": [
        padre
    ]

}


print()

print(
    "Secciones:",
    len(
        padre["secciones"]
    )
)

for s in padre["secciones"]:

    print()

    print(
        s["titulo"]
    )

    print(
        "Descripción:",
        len(
            s["descripcion"]
        )
    )

    print(
        "Tarjetas:",
        len(
            s["tarjetas"]
        )
    )

    print(
        "Acordeones:",
        len(
            s["acordeones"]
        )
    )

    print(
        "Banners:",
        len(
            s["banners"]
        )
    )

    print(
        "Enlaces:",
        len(
            s["enlaces"]
        )
    )


Secciones: 6

Conoce la Escuela de Doctorado
Descripción: 0
Tarjetas: 1
Acordeones: 0
Banners: 0
Enlaces: 10

Elige Programa de Doctorado y confirma tutor
Descripción: 0
Tarjetas: 1
Acordeones: 0
Banners: 1
Enlaces: 1

Consulta las ayudas predoctorales
Descripción: 0
Tarjetas: 1
Acordeones: 0
Banners: 1
Enlaces: 1

Presenta tu solicitud
Descripción: 0
Tarjetas: 1
Acordeones: 0
Banners: 1
Enlaces: 4

Realiza la matrícula
Descripción: 0
Tarjetas: 1
Acordeones: 0
Banners: 1
Enlaces: 6

Comienza tu aventura
Descripción: 0
Tarjetas: 1
Acordeones: 0
Banners: 0
Enlaces: 5


In [7]:
# ==========================================================
# BLOQUE 7. Guardado JSON
# ==========================================================

with open(

    RUTA_JSON,

    "w",

    encoding="utf-8"

) as f:

    json.dump(

        datos,

        f,

        ensure_ascii=False,

        indent=4

    )


print()

print(
    "JSON guardado en:"
)

print(
    RUTA_JSON
)


JSON guardado en:
/content/drive/MyDrive/TFG Teleco/JSONs/admision_doctorado.json


In [8]:
# ==========================================================
# BLOQUE 8. Validación del JSON
# ==========================================================

print("=" * 70)
print("VALIDACIÓN JSON ADMISIÓN DOCTO")
print("=" * 70)

for padre in datos["padres"]:

    print()
    print("PADRE")
    print("-" * 70)

    print("Título:", padre["titulo"])
    print("URL:", padre["url"])

    print()

    for seccion in padre["secciones"]:

        print(seccion["titulo"])

        print(
            "  descripción:",
            len(seccion["descripcion"])
        )

        print(
            "  tarjetas:",
            len(seccion["tarjetas"])
        )

        print(
            "  acordeones:",
            len(seccion["acordeones"])
        )

        print(
            "  banners:",
            len(seccion["banners"])
        )

        print(
            "  enlaces:",
            len(seccion["enlaces"])
        )

        print()

VALIDACIÓN JSON ADMISIÓN DOCTO

PADRE
----------------------------------------------------------------------
Título: Admisión a doctorado
URL: https://www.upv.es/admision/admision-doctorado/index-es.html

Conoce la Escuela de Doctorado
  descripción: 0
  tarjetas: 1
  acordeones: 0
  banners: 0
  enlaces: 10

Elige Programa de Doctorado y confirma tutor
  descripción: 0
  tarjetas: 1
  acordeones: 0
  banners: 1
  enlaces: 1

Consulta las ayudas predoctorales
  descripción: 0
  tarjetas: 1
  acordeones: 0
  banners: 1
  enlaces: 1

Presenta tu solicitud
  descripción: 0
  tarjetas: 1
  acordeones: 0
  banners: 1
  enlaces: 4

Realiza la matrícula
  descripción: 0
  tarjetas: 1
  acordeones: 0
  banners: 1
  enlaces: 6

Comienza tu aventura
  descripción: 0
  tarjetas: 1
  acordeones: 0
  banners: 0
  enlaces: 5



In [9]:
# ==========================================================
# BLOQUE 9. Generación Markdown semántico para RAG
# Admisión Doctorado + metadatos YAML
# ==========================================================

import json
import os
import re


# ----------------------------------------------------------
# Configuración
# ----------------------------------------------------------

ruta_json = RUTA_JSON

directorio_base = os.path.join(
    ruta_programa,
    "ADMISION",
    "Doctorado"
)

os.makedirs(
    directorio_base,
    exist_ok=True
)


# ----------------------------------------------------------
# Parámetros
# ----------------------------------------------------------

CATEGORIA = "admision"
NIVEL = "doctorado"

INTRO_DOCUMENTO = (
    "Información completa sobre el proceso de admisión "
    "a estudios oficiales de doctorado en la "
    "Universitat Politècnica de València."
)


# ----------------------------------------------------------
# Limpieza nombres
# ----------------------------------------------------------

def limpiar_nombre(nombre):

    nombre = nombre.lower()

    cambios = {
        "á":"a",
        "é":"e",
        "í":"i",
        "ó":"o",
        "ú":"u",
        "ñ":"n"
    }

    for viejo, nuevo in cambios.items():

        nombre = nombre.replace(
            viejo,
            nuevo
        )

    nombre = re.sub(
        r"[^a-z0-9]+",
        "_",
        nombre
    )

    return nombre.strip("_")


# ----------------------------------------------------------
# Metadatos YAML
# ----------------------------------------------------------

def escribir_metadatos(
    f,
    tipo_documento,
    seccion=None
):

    f.write("---\n")

    f.write("fuente: UPV\n")

    f.write(
        f"categoria: {CATEGORIA}\n"
    )

    f.write(
        f"nivel: {NIVEL}\n"
    )

    f.write(
        f"tipo_documento: {tipo_documento}\n"
    )

    if seccion:

        f.write(
            f"seccion: {seccion}\n"
        )

    f.write("---\n\n")


# ----------------------------------------------------------
# Cargar JSON
# ----------------------------------------------------------

with open(
    ruta_json,
    encoding="utf-8"
) as f:

    datos = json.load(f)


contador = 0


# ==========================================================
# Generación
# ==========================================================

for padre in datos["padres"]:

    nombre_padre = limpiar_nombre(
        padre["titulo"]
    )


    # ------------------------------------------------------
    # Documento padre
    # ------------------------------------------------------

    archivo_padre = os.path.join(

        directorio_base,

        f"{nombre_padre}.md"

    )


    with open(
        archivo_padre,
        "w",
        encoding="utf-8"
    ) as f:


        escribir_metadatos(
            f,
            "padre"
        )


        f.write(
            f"# {padre['titulo']}\n\n"
        )


        f.write(
            INTRO_DOCUMENTO
            +
            "\n\n"
        )


        for seccion in padre["secciones"]:


            f.write(
                f"## {seccion['titulo']}\n\n"
            )


            if seccion.get(
                "descripcion"
            ):

                f.write(
                    seccion["descripcion"]
                    +
                    "\n\n"
                )


            # Tarjetas

            for t in seccion.get(
                "tarjetas",
                []
            ):

                f.write(
                    f"### {t['titulo']}\n\n"
                )


                if t.get(
                    "descripcion"
                ):

                    f.write(
                        t["descripcion"]
                        +
                        "\n\n"
                    )


                if t.get(
                    "url"
                ):

                    f.write(
                        f"Más información: {t['url']}\n\n"
                    )


            # Acordeones

            for acc in seccion.get(
                "acordeones",
                []
            ):


                if acc.get(
                    "titulo"
                ):

                    f.write(
                        f"### {acc['titulo']}\n\n"
                    )


                if acc.get(
                    "texto"
                ):

                    f.write(
                        acc["texto"]
                        +
                        "\n\n"
                    )


                for enlace in acc.get(
                    "enlaces",
                    []
                ):

                    f.write(
                        f"- {enlace['texto']}: "
                        f"{enlace['url']}\n"
                    )

                f.write("\n")


            # Banners

            for b in seccion.get(
                "banners",
                []
            ):


                if b.get(
                    "titulo"
                ):

                    f.write(
                        f"### {b['titulo']}\n\n"
                    )


                if b.get(
                    "descripcion"
                ):

                    f.write(
                        b["descripcion"]
                        +
                        "\n\n"
                    )


                if b.get(
                    "url"
                ):

                    f.write(
                        f"Más información: {b['url']}\n\n"
                    )


    contador += 1


    # ------------------------------------------------------
    # Documentos por sección
    # ------------------------------------------------------

    for seccion in padre["secciones"]:


        nombre_seccion = limpiar_nombre(
            seccion["titulo"]
        )


        archivo = os.path.join(
            directorio_base,
            f"{nombre_seccion}.md"
        )


        with open(
            archivo,
            "w",
            encoding="utf-8"
        ) as f:


            escribir_metadatos(
                f,
                "seccion",
                nombre_seccion
            )


            f.write(
                f"# {seccion['titulo']}\n\n"
            )


            f.write(
                f"Proceso: {padre['titulo']}\n\n"
            )


            if seccion.get(
                "descripcion"
            ):

                f.write(
                    seccion["descripcion"]
                    +
                    "\n\n"
                )


            # Tarjetas

            for t in seccion.get(
                "tarjetas",
                []
            ):

                f.write(
                    f"## {t['titulo']}\n\n"
                )


                if t.get(
                    "descripcion"
                ):

                    f.write(
                        t["descripcion"]
                        +
                        "\n\n"
                    )


                if t.get(
                    "url"
                ):

                    f.write(
                        f"Enlace oficial: {t['url']}\n\n"
                    )


            # Acordeones

            for acc in seccion.get(
                "acordeones",
                []
            ):


                if acc.get(
                    "titulo"
                ):

                    f.write(
                        f"## {acc['titulo']}\n\n"
                    )


                if acc.get(
                    "texto"
                ):

                    f.write(
                        acc["texto"]
                        +
                        "\n\n"
                    )


                for enlace in acc.get(
                    "enlaces",
                    []
                ):

                    f.write(
                        f"- {enlace['texto']}: {enlace['url']}\n"
                    )

                f.write("\n")


            # Banners

            for b in seccion.get(
                "banners",
                []
            ):


                if b.get(
                    "titulo"
                ):

                    f.write(
                        f"## {b['titulo']}\n\n"
                    )


                if b.get(
                    "descripcion"
                ):

                    f.write(
                        b["descripcion"]
                        +
                        "\n\n"
                    )


                if b.get(
                    "url"
                ):

                    f.write(
                        f"Enlace oficial: {b['url']}\n\n"
                    )


        contador += 1


print()

print(
    "Markdown generado correctamente."
)

print(
    "Archivos creados:",
    contador
)

print(
    "Ruta:",
    directorio_base
)


Markdown generado correctamente.
Archivos creados: 7
Ruta: /content/drive/MyDrive/TFG Teleco/ADMISION/Doctorado


In [10]:
# ==========================================================
# BLOQUE 10. Descubrimiento de enlaces
# ==========================================================

import json


with open(
    RUTA_JSON,
    encoding="utf-8"
) as f:

    datos = json.load(f)


enlaces = {}



def registrar_enlace(
    url,
    texto,
    seccion
):

    if not url:
        return

    if url in enlaces:
        return

    enlaces[url] = {

        "url": url,

        "texto": texto,

        "seccion_origen": seccion

    }



for padre in datos["padres"]:

    for seccion in padre["secciones"]:

        nombre_seccion = seccion["titulo"]


        # ------------------------------------------
        # Enlaces generales
        # ------------------------------------------

        for enlace in seccion.get(
            "enlaces",
            []
        ):

            registrar_enlace(

                enlace["url"],

                enlace["texto"],

                nombre_seccion

            )


        # ------------------------------------------
        # Tarjetas
        # ------------------------------------------

        for tarjeta in seccion.get(
            "tarjetas",
            []
        ):

            registrar_enlace(

                tarjeta.get("url"),

                tarjeta.get("titulo"),

                nombre_seccion

            )


        # ------------------------------------------
        # Banners
        # ------------------------------------------

        for banner in seccion.get(
            "banners",
            []
        ):

            registrar_enlace(

                banner.get("url"),

                banner.get("titulo"),

                nombre_seccion

            )


        # ------------------------------------------
        # Acordeones
        # ------------------------------------------

        for acordeon in seccion.get(
            "acordeones",
            []
        ):

            for enlace in acordeon.get(
                "enlaces",
                []
            ):

                registrar_enlace(

                    enlace["url"],

                    enlace["texto"],

                    nombre_seccion

                )


lista_enlaces = list(
    enlaces.values()
)


print()

print(
    "Enlaces únicos encontrados:",
    len(lista_enlaces)
)

print()

for enlace in lista_enlaces:

    print(
        enlace["seccion_origen"]
    )

    print(
        "   ",
        enlace["texto"]
    )

    print(
        "   ",
        enlace["url"]
    )

    print()


Enlaces únicos encontrados: 25

Conoce la Escuela de Doctorado
    Más información
    https://www.upv.es/entidades/edoctorado/

Conoce la Escuela de Doctorado
    Más información
    https://www.upv.es/entidades/edoctorado/la-escuela-en-cifras/

Conoce la Escuela de Doctorado
    Más información
    https://www.upv.es/entidades/edoctorado/pasos-para-completar-la-tesis/

Conoce la Escuela de Doctorado
    Más información
    https://www.upv.es/entidades/edoctorado/noticias-2/

Conoce la Escuela de Doctorado
    Más información
    https://www.upv.es/china

Conoce la Escuela de Doctorado
    Tablón de anuncios oficial
    https://www.upv.es/entidades/edoctorado/tablon-de-anuncios-oficial/

Conoce la Escuela de Doctorado
    Servicios que ofrece la Escuela de Doctorado
    https://www.upv.es/entidades/edoctorado/consulta/servicios-que-prestamos/

Conoce la Escuela de Doctorado
    Noticias que te interesan
    https://www.upv.es/entidades/edoctorado/noticias/

Conoce la Escuela de Docto

In [12]:
# ==========================================================
# BLOQUE 11. Descarga y extracción de páginas enlazadas
# Admisión Doctorado
# ==========================================================

import requests

from bs4 import BeautifulSoup

from urllib.parse import urlparse


paginas_extraidas = []


# ----------------------------------------------------------
# Alcance del crawler
# ----------------------------------------------------------

def url_relevante(url):

    url = url.lower()

    return (

        "www.upv.es/entidades/edoctorado/" in url

        or

        "www.upv.es/pls/soalu/" in url

    )


# ----------------------------------------------------------
# Descarga de páginas
# ----------------------------------------------------------

for enlace in lista_enlaces:

    url = enlace["url"]


    print()

    print("=" * 80)

    print(url)


    # ------------------------------------------------------
    # Filtrar URLs fuera del ámbito de Doctorado
    # ------------------------------------------------------

    if not url_relevante(url):

        print(
            "Descartado (fuera del ámbito de Doctorado)"
        )

        continue


    try:

        respuesta = requests.get(

            url,

            headers=HEADERS,

            timeout=20,

            allow_redirects=True

        )

        respuesta.raise_for_status()


    except Exception:

        print(
            "Error al descargar."
        )

        continue


    # ------------------------------------------------------
    # Comprobar URL final
    # ------------------------------------------------------

    dominio = urlparse(
        respuesta.url
    ).netloc.lower()


    if "upv.es" not in dominio:

        print(
            "Descartado (dominio externo)"
        )

        continue


    if any(

        palabra in respuesta.url.lower()

        for palabra in [

            "poliformat",

            "login",

            "shibboleth"

        ]

    ):

        print(
            "Descartado (requiere autenticación)"
        )

        continue


    # ------------------------------------------------------
    # Ignorar archivos
    # ------------------------------------------------------

    if respuesta.url.lower().split("?")[0].endswith(

        (

            ".pdf",

            ".jpg",

            ".jpeg",

            ".png",

            ".gif",

            ".zip",

            ".doc",

            ".docx",

            ".xls",

            ".xlsx"

        )

    ):

        print(
            "Descartado (archivo)"
        )

        continue


    print(
        "URL final:",
        respuesta.url
    )


    soup = BeautifulSoup(

        respuesta.text,

        "html.parser"

    )


    # ------------------------------------------------------
    # Eliminar elementos de plantilla
    # ------------------------------------------------------

    for basura in soup.find_all(

        [

            "script",

            "style",

            "noscript",

            "header",

            "footer",

            "nav",

            "aside"

        ]

    ):

        basura.decompose()


    # ------------------------------------------------------
    # Localizar contenido principal
    # ------------------------------------------------------

    contenido_principal = (

        soup.find("article")

        or soup.find(
            "main",
            class_=lambda x:
                x and "content" in " ".join(x)
        )

        or soup.find(
            id="content"
        )

        or soup.find(
            class_=lambda x:
                x and any(
                    palabra in " ".join(x).lower()
                    for palabra in [
                        "content",
                        "contenido",
                        "page-content"
                    ]
                )
        )

    )


    # Si no se encuentra un contenedor claro,
    # no se utiliza toda la página automáticamente.

    if contenido_principal is None:

        contenido_principal = soup.find("main")


    if contenido_principal is None:

        print(
            "Descartado (no se encontró contenido principal)"
        )

        continue


    # ------------------------------------------------------
    # Título
    # ------------------------------------------------------

    titulo = texto_limpio(

        contenido_principal.find("h1")

    )


    if not titulo:

        titulo = enlace["texto"]


    # ------------------------------------------------------
    # Extracción del contenido
    # ------------------------------------------------------

    bloques = []

    vistos = set()


    for elemento in contenido_principal.find_all(

        [

            "h2",

            "h3",

            "h4",

            "p",

            "li"

        ]

    ):


        texto = texto_limpio(

            elemento

        )


        if len(texto) < 5:

            continue


        if texto in vistos:

            continue


        vistos.add(texto)


        bloques.append(

            texto

        )


    contenido = "\n\n".join(

        bloques

    )


    # ------------------------------------------------------
    # Comprobar que realmente se ha obtenido contenido
    # ------------------------------------------------------

    if len(contenido) < 50:

        print(
            "Descartado (contenido insuficiente)"
        )

        continue


    # ------------------------------------------------------
    # Guardar resultado
    # ------------------------------------------------------

    paginas_extraidas.append({

        "url": url,

        "url_final": respuesta.url,

        "titulo": titulo,

        "seccion_origen":
            enlace["seccion_origen"],

        "contenido": contenido

    })


# ==========================================================
# Resultado
# ==========================================================

print()

print("=" * 80)

print(
    "Páginas útiles:",
    len(paginas_extraidas)
)

print()


for pagina in paginas_extraidas[:5]:

    print("=" * 80)

    print(
        pagina["titulo"]
    )

    print()

    print(
        "Sección:",
        pagina["seccion_origen"]
    )

    print()

    print(
        "URL:",
        pagina["url_final"]
    )

    print()

    print(
        pagina["contenido"][:800]
    )

    print()


https://www.upv.es/entidades/edoctorado/
URL final: https://www.upv.es/entidades/edoctorado/

https://www.upv.es/entidades/edoctorado/la-escuela-en-cifras/
URL final: https://www.upv.es/entidades/edoctorado/la-escuela-en-cifras/

https://www.upv.es/entidades/edoctorado/pasos-para-completar-la-tesis/
URL final: https://www.upv.es/entidades/edoctorado/pasos-para-completar-la-tesis/

https://www.upv.es/entidades/edoctorado/noticias-2/
URL final: https://www.upv.es/entidades/edoctorado/noticias-2/

https://www.upv.es/china
Descartado (fuera del ámbito de Doctorado)

https://www.upv.es/entidades/edoctorado/tablon-de-anuncios-oficial/
URL final: https://www.upv.es/entidades/edoctorado/tablon-de-anuncios-oficial/

https://www.upv.es/entidades/edoctorado/consulta/servicios-que-prestamos/
URL final: https://www.upv.es/entidades/edoctorado/consulta/servicios-que-prestamos/
Descartado (contenido insuficiente)

https://www.upv.es/entidades/edoctorado/noticias/
URL final: https://www.upv.es/entida

In [14]:
# ==========================================================
# BLOQUE 12. Generación de Markdown de páginas enlazadas
# Admisión Doctorado + metadatos YAML
# ==========================================================

import os
import re


# ----------------------------------------------------------
# Configuración
# ----------------------------------------------------------

directorio_base = os.path.join(
    ruta_programa,
    "ADMISION",
    "Doctorado"
)

directorio_recursos = os.path.join(
    directorio_base,
    "recursos"
)

os.makedirs(
    directorio_recursos,
    exist_ok=True
)


CATEGORIA = "admision"
NIVEL = "doctorado"


# ----------------------------------------------------------
# Limpieza de nombres de archivo
# ----------------------------------------------------------

def limpiar_nombre(nombre):

    nombre = nombre.lower()

    cambios = {
        "á": "a",
        "é": "e",
        "í": "i",
        "ó": "o",
        "ú": "u",
        "ñ": "n"
    }

    for viejo, nuevo in cambios.items():

        nombre = nombre.replace(
            viejo,
            nuevo
        )

    nombre = re.sub(
        r"[^a-z0-9]+",
        "_",
        nombre
    )

    return nombre.strip("_")


# ----------------------------------------------------------
# Clasificación del recurso
# ----------------------------------------------------------

def clasificar_recurso(pagina):

    texto = (

        pagina["titulo"]
        + " "
        + pagina["url_final"]

    ).lower()


    # Plazos y calendarios

    if any(

        palabra in texto

        for palabra in [

            "plazo",
            "calendario",
            "calendarios",
            "fechas"

        ]

    ):

        return "calendario"


    # Tasas y precios

    if any(

        palabra in texto

        for palabra in [

            "precio",
            "precios",
            "tasas",
            "matricula"

        ]

    ):

        return "matricula"


    # Preguntas frecuentes

    if any(

        palabra in texto

        for palabra in [

            "faq",
            "faqs",
            "preguntas frecuentes"

        ]

    ):

        return "faq"


    # Ayudas

    if any(

        palabra in texto

        for palabra in [

            "ayuda",
            "ayudas",
            "beca",
            "becas"

        ]

    ):

        return "ayudas"


    # Solicitud / preinscripción

    if any(

        palabra in texto

        for palabra in [

            "solicitud",
            "preinscripcion",
            "preinscripción",
            "admision",
            "admisión"

        ]

    ):

        return "admision"


    # Programas

    if any(

        palabra in texto

        for palabra in [

            "programas-de-doctorado",
            "programa de doctorado",
            "oferta"

        ]

    ):

        return "programas"


    # Normativa

    if any(

        palabra in texto

        for palabra in [

            "normativa",
            "reglamento",
            "legislacion",
            "legislación"

        ]

    ):

        return "normativa"


    # Por defecto

    return "informacion"


# ----------------------------------------------------------
# Metadatos YAML
# ----------------------------------------------------------

def escribir_metadatos(
    f,
    pagina,
    tipo_recurso
):

    f.write(
        "---\n"
    )

    f.write(
        "fuente: UPV\n"
    )

    f.write(
        f"categoria: {CATEGORIA}\n"
    )

    f.write(
        f"nivel: {NIVEL}\n"
    )

    f.write(
        "tipo_documento: recurso\n"
    )

    f.write(
        f"tipo_recurso: {tipo_recurso}\n"
    )

    f.write(
        "seccion_origen: "
        + limpiar_nombre(
            pagina["seccion_origen"]
        )
        + "\n"
    )

    f.write(
        f"url: {pagina['url_final']}\n"
    )

    f.write(
        "---\n\n"
    )


# ==========================================================
# Eliminación de duplicados
# ==========================================================

paginas_unicas = []

urls_vistas = set()


for pagina in paginas_extraidas:

    url_final = pagina["url_final"]


    if url_final in urls_vistas:

        continue


    urls_vistas.add(
        url_final
    )

    paginas_unicas.append(
        pagina
    )


# ==========================================================
# Generación de Markdown
# ==========================================================

contador = 0


for pagina in paginas_unicas:


    # ------------------------------------------------------
    # Clasificar recurso
    # ------------------------------------------------------

    tipo_recurso = clasificar_recurso(
        pagina
    )


    # ------------------------------------------------------
    # Nombre del archivo
    # ------------------------------------------------------

    nombre = limpiar_nombre(
        pagina["titulo"]
    )


    if not nombre:

        nombre = "recurso"


    archivo = os.path.join(

        directorio_recursos,

        f"{nombre}.md"

    )


    # ------------------------------------------------------
    # Evitar colisiones de nombres
    # ------------------------------------------------------

    if os.path.exists(
        archivo
    ):

        nombre_seccion = limpiar_nombre(
            pagina["seccion_origen"]
        )

        archivo = os.path.join(

            directorio_recursos,

            f"{nombre}_{nombre_seccion}.md"

        )


    # ------------------------------------------------------
    # Escritura
    # ------------------------------------------------------

    with open(

        archivo,

        "w",

        encoding="utf-8"

    ) as f:


        escribir_metadatos(

            f,

            pagina,

            tipo_recurso

        )


        f.write(
            f"# {pagina['titulo']}\n\n"
        )


        f.write(
            "Contenido relacionado con el "
            "proceso de admisión a doctorado.\n\n"
        )


        f.write(
            f"Sección de origen: "
            f"{pagina['seccion_origen']}\n\n"
        )


        f.write(
            pagina["contenido"]
            +
            "\n\n"
        )


        f.write(
            f"Fuente oficial: "
            f"{pagina['url_final']}\n"
        )


    contador += 1


# ==========================================================
# Resultado
# ==========================================================

print()

print("=" * 70)

print(
    "Markdown de recursos generado correctamente."
)

print(
    "Páginas descargadas:",
    len(paginas_extraidas)
)

print(
    "Páginas únicas:",
    len(paginas_unicas)
)

print(
    "Archivos creados:",
    contador
)

print()

print(
    "Ruta:",
    directorio_recursos
)


Markdown de recursos generado correctamente.
Páginas descargadas: 18
Páginas únicas: 17
Archivos creados: 17

Ruta: /content/drive/MyDrive/TFG Teleco/ADMISION/Doctorado/recursos
